# Hopf Reservoir Computer — End-to-End Pipeline Demo

This notebook walks through the full pipeline from Hopf oscillator simulation
to CMSIS-NN-ready INT8 TFLite model.

References:
- "A Hopf physical reservoir computer" (Shougat et al., Scientific Reports 2021)
- "Hopf physical reservoir computer for reconfigurable sound recognition" (Shougat et al., Scientific Reports 2023)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

## 1. Generate Synthetic Hopf Oscillator Data

In [ ]:
from data.sample_data import generate_dataset, CLASS_NAMES

# Use a small dataset for the demo (10 clips per class)
raw_x, labels = generate_dataset(n_clips_per_class=10, n_classes=5, cache=False)
print(f"Raw data shape: {raw_x.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Classes: {CLASS_NAMES}")

In [ ]:
# Plot raw x(t) for one clip per class
fig, axes = plt.subplots(5, 1, figsize=(12, 10), sharex=True)
for cls in range(5):
    idx = np.where(labels == cls)[0][0]
    t = np.arange(raw_x.shape[1]) / 100_000  # time in seconds
    axes[cls].plot(t[:4000], raw_x[idx, :4000], linewidth=0.5)
    axes[cls].set_ylabel(CLASS_NAMES[cls])
axes[-1].set_xlabel('Time (s)')
fig.suptitle('Raw Hopf Oscillator x(t) — First 40ms per Class')
plt.tight_layout()
plt.show()

## 2. Ingest: Downsample, Normalise, atanh Activation

In [ ]:
from pipeline.ingest import process_dataset

processed = process_dataset(raw_x)
print(f"Processed shape: {processed.shape}  (n_clips, time_steps, virtual_nodes)")
print(f"Value range: [{processed.min():.3f}, {processed.max():.3f}]")

## 3. Feature Extraction: uint8 Scaling + Visualisation

In [ ]:
from pipeline.features import extract_features, visualise_features

feature_maps, labels_out = extract_features(processed, labels)
print(f"Feature maps: {feature_maps.shape}, dtype={feature_maps.dtype}")

visualise_features(feature_maps, labels_out, CLASS_NAMES)

## 4. Build CMSIS-NN-Compatible CNN

In [ ]:
from pipeline.model import build_model, print_model_summary

model = build_model(n_classes=5)
print_model_summary(model)

## 5. Quantisation-Aware Training

In [ ]:
from pipeline.train import apply_qat, prepare_data, train

# Apply QAT
try:
    qat_model = apply_qat(model)
    print("QAT applied successfully")
except ImportError:
    print("tensorflow-model-optimization not installed, using base model")
    qat_model = model

# Prepare data
x_train, y_train, x_val, y_val = prepare_data(feature_maps, labels_out, n_classes=5)
print(f"Train: {x_train.shape}, Val: {x_val.shape}")

# Train
history = train(qat_model, x_train, y_train, x_val, y_val, epochs=10, batch_size=8)

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['loss'], label='train')
ax1.plot(history.history['val_loss'], label='val')
ax1.set_title('Loss'); ax1.legend()
ax2.plot(history.history['accuracy'], label='train')
ax2.plot(history.history['val_accuracy'], label='val')
ax2.set_title('Accuracy'); ax2.legend()
plt.tight_layout()
plt.show()

## 6. Convert to TFLite INT8 + Generate CMSIS-NN Artefacts

In [ ]:
from pipeline.convert import convert_pipeline
from pipeline.train import representative_data_gen

def rep_gen():
    return representative_data_gen(feature_maps, n_samples=50)

convert_pipeline(qat_model, rep_gen, firmware_dir='../firmware')

## Done

Generated artefacts in `firmware/`:
- `model.tflite` — INT8 quantised TFLite model
- `cmsis_nn_params.h` — weights, biases, quantisation parameters
- `model_data.cc` — TFLite Micro C array
- `deployment_notes.md` — per-MCU compiler flags and deployment instructions